<a href="https://colab.research.google.com/github/ac1d301/clevercsv_speedup/blob/main/candidate_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       858Mi       9.1Gi       7.0Mi       2.7Gi        11Gi
Swap:             0B          0B          0B
Python 3.12.13


In [2]:
import os

if not os.path.exists('/content/CleverCSV'):
    !git clone https://github.com/alan-turing-institute/CleverCSV.git
    %cd /content/CleverCSV
    !git checkout ae043c948fd03eea2ae726525c4f347646d22316
    %cd /content
else:
    print("CleverCSV already cloned")

if not os.path.exists('/content/clevercsv_speedup'):
    !git clone https://github.com/ac1d301/clevercsv_speedup.git

!pip install pybind11 -q
print("Setup complete")

CleverCSV already cloned
Setup complete


In [3]:
%%writefile /content/type_detector.cpp
#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <string>
#include <vector>
#include <cctype>

namespace py = pybind11;

static inline bool is_empty(const std::string& s) { return s.empty(); }

static inline bool is_nan(const std::string& s) {
    return s == "nan" || s == "na" || s == "n/a" ||
           s == "NaN" || s == "NA" || s == "N/A" || s == "NAN";
}

static inline bool is_number(const std::string& s) {
    if (s.empty()) return false;
    size_t i = 0;
    if (s[i] == '+' || s[i] == '-') i++;
    if (i == s.size()) return false;
    bool has_digit = false;
    while (i < s.size() && std::isdigit((unsigned char)s[i])) { has_digit = true; i++; }
    if (i < s.size() && s[i] == '.') {
        i++;
        while (i < s.size() && std::isdigit((unsigned char)s[i])) { has_digit = true; i++; }
    }
    if (!has_digit) return false;
    if (i < s.size() && (s[i] == 'e' || s[i] == 'E')) {
        i++;
        if (i < s.size() && (s[i] == '+' || s[i] == '-')) i++;
        if (i == s.size()) return false;
        while (i < s.size() && std::isdigit((unsigned char)s[i])) i++;
    }
    return i == s.size();
}

static inline bool is_number_with_commas(const std::string& s) {
    if (s.empty()) return false;
    size_t i = 0;
    if (s[i] == '+' || s[i] == '-') i++;
    bool has_digit = false;
    while (i < s.size() && (std::isdigit((unsigned char)s[i]) || s[i] == ',')) {
        if (std::isdigit((unsigned char)s[i])) has_digit = true;
        i++;
    }
    if (i < s.size() && s[i] == '.') {
        i++;
        while (i < s.size() && std::isdigit((unsigned char)s[i])) i++;
    }
    return has_digit && i == s.size();
}

static inline bool is_percentage(const std::string& s) {
    if (s.empty() || s.back() != '%') return false;
    std::string num = s.substr(0, s.size() - 1);
    return is_number(num) || is_number_with_commas(num);
}

static inline bool is_currency(const std::string& s) {
    if (s.empty()) return false;
    if (s[0] != '$') return false;
    return is_number(s.substr(1)) || is_number_with_commas(s.substr(1));
}

static inline bool is_ipv4(const std::string& s) {
    int dots = 0, num = 0, digits = 0;
    for (char c : s) {
        if (std::isdigit((unsigned char)c)) {
            num = num * 10 + (c - '0');
            digits++;
            if (num > 255 || digits > 3) return false;
        } else if (c == '.') {
            if (digits == 0) return false;
            dots++; num = 0; digits = 0;
        } else return false;
    }
    return dots == 3 && digits > 0;
}

static bool is_date_str(const std::string& s) {
    if (s.empty() || !std::isdigit((unsigned char)s[0])) return false;
    int digits = 0, seps = 0;
    char sep = 0;
    for (char c : s) {
        if (std::isdigit((unsigned char)c)) digits++;
        else if (c == '-' || c == '/' || c == '.') {
            if (sep == 0) sep = c;
            else if (sep != c) return false;
            seps++;
        } else return false;
    }
    return digits >= 4 && digits <= 8 && seps == 2;
}

static bool is_time_str(const std::string& s) {
    if (s.empty() || !std::isdigit((unsigned char)s[0])) return false;
    int digits = 0, colons = 0;
    for (char c : s) {
        if (std::isdigit((unsigned char)c)) digits++;
        else if (c == ':') colons++;
        else if (c == '.' || c == 'Z' || c == '+' || c == '-') { }
        else return false;
    }
    return digits >= 4 && colons >= 1 && colons <= 2;
}

static inline bool is_datetime(const std::string& s) {
    if (s.empty() || !std::isdigit((unsigned char)s[0])) return false;
    size_t sep = s.find(' ');
    if (sep == std::string::npos) sep = s.find('T');
    if (sep == std::string::npos) return false;
    return is_date_str(s.substr(0, sep)) && is_time_str(s.substr(sep + 1));
}

static inline bool is_url(const std::string& s) {
    if (s.size() < 7) return false;
    if (s.substr(0, 7) == "http://") return true;
    if (s.size() >= 8 && s.substr(0, 8) == "https://") return true;
    if (s.substr(0, 6) == "ftp://") return true;
    return false;
}

static inline bool is_email(const std::string& s) {
    size_t at = s.find('@');
    if (at == std::string::npos || at == 0 || at == s.size()-1) return false;
    size_t dot = s.find('.', at);
    return dot != std::string::npos && dot != s.size()-1;
}

static inline bool is_unix_path(const std::string& s) { return !s.empty() && s[0] == '/'; }

static inline bool is_bytearray(const std::string& s) {
    return s.size() > 12 && s.substr(0, 11) == "bytearray(b" && s.back() == ')';
}

static inline bool is_json_obj(const std::string& s) {
    return !s.empty() && s.front() == '{' && s.back() == '}';
}

static inline bool is_unicode_alphanum(const std::string& s) {
    if (s.empty()) return false;
    for (unsigned char c : s) {
        if (c >= 128) return true;
        if (!std::isalnum(c) && c != ' ' && c != '_' && c != '-' && c != '.') return false;
    }
    return true;
}

static bool is_known_type(const std::string& cell, bool is_quoted) {
    size_t start = 0, end = cell.size();
    while (start < end && std::isspace((unsigned char)cell[start])) start++;
    while (end > start && std::isspace((unsigned char)cell[end-1])) end--;
    std::string s = cell.substr(start, end - start);
    if (is_empty(s))          return true;
    if (is_url(s))            return true;
    if (is_email(s))          return true;
    if (is_ipv4(s))           return true;
    if (is_number(s))         return true;
    if (is_number_with_commas(s)) return true;
    if (is_time_str(s))       return true;
    if (is_percentage(s))     return true;
    if (is_currency(s))       return true;
    if (is_unix_path(s))      return true;
    if (is_nan(s))            return true;
    if (is_date_str(s))       return true;
    if (is_datetime(s))       return true;
    if (is_unicode_alphanum(s)) return true;
    if (is_bytearray(s))      return true;
    if (is_json_obj(s))       return true;
    return false;
}

double bulk_type_score(
    const std::vector<std::vector<std::pair<std::string, bool>>>& rows,
    double eps = 1e-10
) {
    long long total = 0, known = 0;
    for (const auto& row : rows) {
        for (const auto& cell_pair : row) {
            total++;
            if (is_known_type(cell_pair.first, cell_pair.second)) known++;
        }
    }
    if (total == 0) return eps;
    double score = static_cast<double>(known) / total;
    return score < eps ? eps : score;
}

PYBIND11_MODULE(type_detector, m) {
    m.def("bulk_type_score", &bulk_type_score, py::arg("rows"), py::arg("eps") = 1e-10);
    m.def("is_known_type", &is_known_type, py::arg("cell"), py::arg("is_quoted") = false);
}

Overwriting /content/type_detector.cpp


In [ ]:
import sysconfig, subprocess, sys

includes = subprocess.check_output(
    [sys.executable, '-m', 'pybind11', '--includes']
).decode().strip()
ext_suffix = sysconfig.get_config_var('EXT_SUFFIX')

!rm -f /content/type_detector*.so

cmd = (f"g++ -O3 -shared -fPIC -std=c++17 -Wno-multichar "
       f"{includes} /content/type_detector.cpp -o /content/type_detector{ext_suffix}")

print(f"Running: {cmd}")
ret = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(ret.stderr)
print("Return code:", ret.returncode)
!ls -la /content/type_detector*.so

In [4]:
import re

cons_path = '/content/CleverCSV/clevercsv/consistency.py'

# Restore original first
%cd /content/CleverCSV
!git checkout clevercsv/consistency.py
%cd /content

with open(cons_path) as f:
    content = f.read()

new_compute_type_score = '''    def compute_type_score(
        self, data: str, dialect: SimpleDialect, eps: float = DEFAULT_EPS_TYPE
    ) -> float:
        """Compute the type score — patched to use C++ extension (v1)."""
        import sys
        if '/content' not in sys.path:
            sys.path.insert(0, '/content')
        import type_detector as _td_cpp
        rows = list(parse_string(data, dialect, return_quoted=True))
        return _td_cpp.bulk_type_score(rows, eps)
'''

pattern = r'    def compute_type_score\(.*?return max\(eps, known / total\)\n'
new_content = re.sub(pattern, new_compute_type_score, content, count=1, flags=re.DOTALL)

if new_content == content:
    print("ERROR: pattern not matched")
else:
    with open(cons_path, 'w') as f:
        f.write(new_content)
    print("✓ Patched consistency.py")

/content/CleverCSV
Updated 1 path from the index
/content
✓ Patched consistency.py


In [5]:
%cd /content/CleverCSV
!pip install -e . -q
%cd /content
print("Installed")

/content/CleverCSV
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for clevercsv (pyproject.toml) ... done
/content
Installed


In [6]:
import sys
for m in list(sys.modules):
    if 'clevercsv' in m or 'type_detector' in m:
        del sys.modules[m]

sys.path.insert(0, '/content')

import clevercsv
from clevercsv.consistency import ConsistencyDetector
import inspect

src = inspect.getsource(ConsistencyDetector.compute_type_score)
print("Patched live:", "type_detector" in src and "bulk_type_score" in src)
print("Installed from:", clevercsv.__file__)

Patched live: True
Installed from: /content/CleverCSV/clevercsv/__init__.py


In [7]:
import time, statistics

data_path = "/content/clevercsv_speedup/data/fec_sample.txt"
with open(data_path, encoding='latin-1') as f:
    raw = f.read()
data = '\n'.join(raw.split('\n')[:25000])
print(f"Loaded {len(data.split(chr(10)))} rows, {len(data)/1e6:.2f} MB")

print("\nWarming up...")
for i in range(2):
    t0 = time.perf_counter()
    dialect = clevercsv.Detector().detect(data)
    print(f"  Warmup {i+1}: {time.perf_counter()-t0:.2f}s")

print("\nMeasuring...")
times = []
for i in range(5):
    t0 = time.perf_counter()
    dialect = clevercsv.Detector().detect(data)
    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    print(f"  Run {i+1}: {elapsed:.2f}s")

times_sorted = sorted(times)
median = statistics.median(times)
iqr = times_sorted[3] - times_sorted[1]

print(f"\nCandidate median: {median:.2f}s")
print(f"Candidate IQR:    {iqr:.2f}s")
print(f"Baseline median:  20.30s")
print(f"Speedup:          {20.30/median:.2f}x")
print(f"Dialect:          {dialect}")

Loaded 25000 rows, 4.81 MB

Warming up...
  Warmup 1: 16.52s
  Warmup 2: 16.02s

Measuring...
  Run 1: 14.70s
  Run 2: 14.85s
  Run 3: 14.34s
  Run 4: 14.44s
  Run 5: 14.93s

Candidate median: 14.70s
Candidate IQR:    0.42s
Baseline median:  20.30s
Speedup:          1.38x
Dialect:          SimpleDialect('|', '', '')


In [8]:
import json

candidate_result = {
    'median_time': median,
    'iqr': iqr,
    'speedup': 20.30/median,
    'baseline_median': 20.30,
    'dialect': str(dialect),
    'n_warmup': 2,
    'n_measured': 5,
    'n_rows': 25000,
    'all_times': times
}

with open('candidate_output.json', 'w') as f:
    json.dump(candidate_result, f, indent=2)

print("Saved candidate_output.json")
print(candidate_result)

Saved candidate_output.json
{'median_time': 14.695959565000521, 'iqr': 0.41617593800037866, 'speedup': 1.3813320532227036, 'baseline_median': 20.3, 'dialect': "SimpleDialect('|', '', '')", 'n_warmup': 2, 'n_measured': 5, 'n_rows': 25000, 'all_times': [14.695959565000521, 14.853500905999681, 14.343717970999933, 14.437324967999302, 14.92570353300107]}
